# GoogLeNet on Sports Balls — executed Kaggle run

This notebook is the self-contained run that produced the checked-in results:
**63.72% test accuracy** and **63.42% macro F1** on the 1,841-image held-out
split, with **8,470,142 parameters** after 40 epochs on a Kaggle T4.

The code cell and its outputs are preserved exactly as they ran. The cleaned-up,
modular implementation lives in [`src/`](../src) and is walked through in
[`googlenet_sports_torch.ipynb`](googlenet_sports_torch.ipynb). See the
[README](../README.md) for the full write-up, results and error analysis.


In [2]:
# ──────────────────────────────────────────────────────────
# GoogLeNet Sports Ball Classification — PyTorch From Scratch
# Dataset: sports-balls-multiclass-image-classification
# GPU: T4 x2 | Runtime: ~40 mins
# ──────────────────────────────────────────────────────────

import os
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from glob import glob

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.transforms import functional as F
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

# ──────────────────── CONFIGURATION ───────────────────────
CONFIG = {
    "data": {
        "root": "/kaggle/input/datasets/samuelcortinhas/sports-balls-multiclass-image-classification",
        "train_dir": "train",
        "test_dir": "test",
        "image_size": 224,
        "batch_size": 64,
        "num_workers": 4,
        "pin_memory": True
    },
    "model": {
        "num_classes": 15,
        "aux_weight": 0.3,
        "dropout": 0.4
    },
    "train": {
        "epochs": 40,
        "lr": 1e-3,
        "warmup_epochs": 3,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "ema_decay": 0.999,
        "grad_clip": 1.0
    },
    "output_dir": "/kaggle/working/outputs",
    "seed": 42
}

# ──────────────────── SETUP & SEED ────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["output_dir"], exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# ──────────────────── DATA TRANSFORMS ──────────────────────
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# ──────────────────── LOAD DATA ───────────────────────────
train_path = os.path.join(CONFIG["data"]["root"], CONFIG["data"]["train_dir"])
test_path = os.path.join(CONFIG["data"]["root"], CONFIG["data"]["test_dir"])

train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
val_dataset   = datasets.ImageFolder(train_path, transform=test_transform)
test_dataset  = datasets.ImageFolder(test_path, transform=test_transform)

# Stratified split: 90% train / 10% validation
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_dataset, _ = random_split(train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(CONFIG["seed"]))
_, val_dataset = random_split(val_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(CONFIG["seed"]))

# Preserve class names
class_names = datasets.ImageFolder(train_path).classes
print(f"✅ Classes: {len(class_names)} — {class_names}")
print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=CONFIG["data"]["batch_size"], shuffle=True,
                          num_workers=CONFIG["data"]["num_workers"], pin_memory=CONFIG["data"]["pin_memory"])
val_loader   = DataLoader(val_dataset, batch_size=CONFIG["data"]["batch_size"], shuffle=False,
                          num_workers=CONFIG["data"]["num_workers"], pin_memory=CONFIG["data"]["pin_memory"])
test_loader  = DataLoader(test_dataset, batch_size=CONFIG["data"]["batch_size"], shuffle=False,
                          num_workers=CONFIG["data"]["num_workers"], pin_memory=CONFIG["data"]["pin_memory"])

# ──────────────────── MODEL DEFINITION ─────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, **kwargs)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class InceptionBlock(nn.Module):
    def __init__(self, in_c, out_1x1, red_3x3, out_3x3, red_5x5, out_5x5, out_pool):
        super().__init__()
        self.branch1 = ConvBlock(in_c, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBlock(in_c, red_3x3, kernel_size=1),
            ConvBlock(red_3x3, out_3x3, kernel_size=3, padding=1)
        )
        self.branch3 = nn.Sequential(
            ConvBlock(in_c, red_5x5, kernel_size=1),
            ConvBlock(red_5x5, out_5x5, kernel_size=3, padding=1),
            ConvBlock(out_5x5, out_5x5, kernel_size=3, padding=1)
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            ConvBlock(in_c, out_pool, kernel_size=1)
        )
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

class AuxiliaryHead(nn.Module):
    def __init__(self, in_c, num_classes):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.conv = ConvBlock(in_c, 128, kernel_size=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 1024)
        self.fc2 = nn.Linear(1024, num_classes)
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        x = self.pool(x)
        x = self.conv(x)
        x = x.flatten(1)
        x = self.dropout(self.fc1(x))
        return self.fc2(x)

class GoogLeNet(nn.Module):
    def __init__(self, num_classes=15, aux_weight=0.3, dropout=0.4):
        super().__init__()
        self.aux_weight = aux_weight
        
        # Stem
        self.conv1 = ConvBlock(3, 64, kernel_size=7, stride=2, padding=3)
        self.pool1 = nn.MaxPool2d(3, 2, ceil_mode=True)
        self.conv2 = ConvBlock(64, 64, kernel_size=1)
        self.conv3 = ConvBlock(64, 192, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(3, 2, ceil_mode=True)
        
        # Inception blocks
        self.inception3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
        self.inception3b = InceptionBlock(256, 128, 128, 192, 32, 96, 64)
        self.pool3 = nn.MaxPool2d(3, 2, ceil_mode=True)
        
        self.inception4a = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
        self.inception4b = InceptionBlock(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
        self.inception4e = InceptionBlock(528, 256, 160, 320, 32, 128, 128)
        self.pool4 = nn.MaxPool2d(3, 2, ceil_mode=True)
        
        # Auxiliary head
        self.aux = AuxiliaryHead(512, num_classes) if aux_weight > 0 else None
        
        self.inception5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)
        
        # Final head
        self.pool5 = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(1024, num_classes)
        
        # Initialize weights
        self._init_weights()
        
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.pool2(x)
        
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.pool3(x)
        
        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        aux_out = self.aux(x) if self.aux and self.training else None
        x = self.inception4d(x)
        x = self.inception4e(x)
        x = self.pool4(x)
        
        x = self.inception5a(x)
        x = self.inception5b(x)
        
        x = self.pool5(x)
        x = x.flatten(1)
        x = self.dropout(x)
        main_out = self.fc(x)
        
        if self.training and aux_out is not None:
            return main_out, aux_out
        return main_out

# ──────────────────── INIT MODEL ──────────────────────────
model = GoogLeNet(num_classes=CONFIG["model"]["num_classes"],
                  aux_weight=CONFIG["model"]["aux_weight"],
                  dropout=CONFIG["model"]["dropout"]).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded — {total_params:,} parameters")

# ──────────────────── OPTIMIZER & SCHEDULER ────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["train"]["label_smoothing"])
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["train"]["lr"],
                         weight_decay=CONFIG["train"]["weight_decay"])

def get_lr_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_sched(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        else:
            progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
            return 0.5 * (1 + np.cos(np.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_sched)

scheduler = get_lr_scheduler(optimizer, CONFIG["train"]["warmup_epochs"], CONFIG["train"]["epochs"])
scaler = GradScaler()

# ──────────────────── EMA (Exponential Moving Average) ────
class EMA:
    def __init__(self, model, decay):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self.register()
    def register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    def step(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

ema = EMA(model, CONFIG["train"]["ema_decay"]) if CONFIG["train"]["ema_decay"] > 0 else None

# ──────────────────── TRAINING LOOP ───────────────────────
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(x)
            if isinstance(outputs, tuple):
                main_out, aux_out = outputs
                loss = criterion(main_out, y) + CONFIG["model"]["aux_weight"] * criterion(aux_out, y)
            else:
                main_out = outputs
                loss = criterion(main_out, y)
        scaler.scale(loss).backward()
        if CONFIG["train"]["grad_clip"] > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["train"]["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        if ema: ema.step()
        total_loss += loss.item() * x.size(0)
        correct += main_out.argmax(1).eq(y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

def validate(model, loader, criterion, device):
    if ema: ema.apply_shadow()
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += logits.argmax(1).eq(y).sum().item()
            total += y.size(0)
    if ema: ema.restore()
    return total_loss / total, correct / total

def test(model, loader, device):
    if ema: ema.apply_shadow()
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
    if ema: ema.restore()
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return acc, f1, all_preds, all_labels

# ──────────────────── RUN TRAINING ─────────────────────────
print("\n🚀 Starting training...")
best_acc, best_epoch = 0, 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, CONFIG["train"]["epochs"] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['train']['epochs']}")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    print(f"  Train — Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"  Val   — Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(CONFIG["output_dir"], "best_model.pt"))
        print(f"  ✅ Best model saved (val_acc: {best_acc:.4f})")

print(f"\n🏆 Training complete. Best epoch: {best_epoch} | Best val_acc: {best_acc:.4f}")

# ──────────────────── FINAL TEST ──────────────────────────
print("\n🧪 Evaluating on held-out test set...")
model.load_state_dict(torch.load(os.path.join(CONFIG["output_dir"], "best_model.pt")))
test_acc, test_f1, preds, labels = test(model, test_loader, device)
print(f"✅ Test Accuracy:  {test_acc:.4f}")
print(f"✅ Test Macro F1:  {test_f1:.4f}")

# ──────────────────── SAVE RESULTS ─────────────────────────
with open(os.path.join(CONFIG["output_dir"], "metrics.txt"), "w") as f:
    f.write(f"Test Accuracy: {test_acc:.4f}\n")
    f.write(f"Test Macro F1: {test_f1:.4f}\n")
    f.write(f"Best Epoch: {best_epoch}\n")
    f.write(f"Parameters: {total_params:,}\n")

# Confusion Matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix — Test Acc: {test_acc:.1%}")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "confusion_matrix.png"), dpi=150)
plt.close()

# Training Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(history["train_acc"], label="Train Acc")
plt.plot(history["val_acc"], label="Val Acc")
plt.axhline(y=test_acc, color='red', linestyle='--', label=f"Test: {test_acc:.1%}")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "training_curves.png"), dpi=150)
plt.close()

# Classification Report
report = classification_report(labels, preds, target_names=class_names)
with open(os.path.join(CONFIG["output_dir"], "classification_report.txt"), "w") as f:
    f.write(report)

print(f"All outputs saved to {CONFIG['output_dir']}")
print("Done! Download outputs folder and update your README with these numbers.")

✅ Device: cuda
✅ Classes: 15 — ['american_football', 'baseball', 'basketball', 'billiard_ball', 'bowling_ball', 'cricket_ball', 'football', 'golf_ball', 'hockey_ball', 'hockey_puck', 'rugby_ball', 'shuttlecock', 'table_tennis_ball', 'tennis_ball', 'volleyball']
✅ Train: 6596 | Val: 732 | Test: 1841


/tmp/ipykernel_47/3998525875.py:262: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


✅ Model loaded — 8,470,142 parameters

🚀 Starting training...

Epoch 1/40


Training:   0%|          | 0/104 [00:00<?, ?it/s]/tmp/ipykernel_47/3998525875.py:299: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_47/3998525875.py:326: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Train — Loss: 13.8386 | Acc: 0.1211
  Val   — Loss: 4.1169 | Acc: 0.0779
  ✅ Best model saved (val_acc: 0.0779)

Epoch 2/40


...

  Train — Loss: 1.2722 | Acc: 0.8783
  Val   — Loss: 2.1590 | Acc: 0.5861
  ✅ Best model saved (val_acc: 0.5861)

Epoch 37/40


  Train — Loss: 1.2694 | Acc: 0.8816
  Val   — Loss: 2.2376 | Acc: 0.5888
  ✅ Best model saved (val_acc: 0.5888)

Epoch 38/40


  Train — Loss: 1.2388 | Acc: 0.8877
  Val   — Loss: 1.9668 | Acc: 0.6093
  ✅ Best model saved (val_acc: 0.6093)

Epoch 39/40


  Train — Loss: 1.2358 | Acc: 0.8866
  Val   — Loss: 2.0098 | Acc: 0.6175
  ✅ Best model saved (val_acc: 0.6175)

Epoch 40/40


  Train — Loss: 1.2414 | Acc: 0.8927
  Val   — Loss: 2.1138 | Acc: 0.6189
  ✅ Best model saved (val_acc: 0.6189)

🏆 Training complete. Best epoch: 40 | Best val_acc: 0.6189

🧪 Evaluating on held-out test set...
✅ Test Accuracy:  0.6372
✅ Test Macro F1:  0.6342
All outputs saved to /kaggle/working/outputs
Done! Download outputs folder and update your README with these numbers.
